In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from google.colab import files
import os

# ----------------------------------
# Upload CSV file in Colab
# ----------------------------------

uploaded = files.upload()   # Upload intern_level_cleaned.csv

# Get uploaded filename automatically
csv_filename = list(uploaded.keys())[0]
print("Uploaded file:", csv_filename)

# ----------------------------------
# Load Dataset
# ----------------------------------

df = pd.read_csv(csv_filename)

# Remove unwanted spaces in column names
df.columns = df.columns.str.strip()

print("Available Columns:\n", df.columns)

# ----------------------------------
# 1. Task Completion Rate
# ----------------------------------

df["task_completion_rate"] = np.where(
    df["total_tasks_assigned"] > 0,
    df["total_tasks_completed"] / df["total_tasks_assigned"],
    0
)

df["task_completion_rate"] = df["task_completion_rate"].clip(0, 1)

# ----------------------------------
# 2. Delay Score
# ----------------------------------

df["delay_score"] = np.where(
    df["weeks_completed"] > 0,
    1 / (1 + (df["total_late_submissions"] / df["weeks_completed"])),
    0
)

# ----------------------------------
# 3. Normalize Features
# ----------------------------------

scaler = MinMaxScaler()

required_columns = ["avg_attendance", "avg_mentor_rating"]

if "avg_feedback_score" in df.columns:
    required_columns.append("avg_feedback_score")
    feedback_exists = True
else:
    print("⚠ avg_feedback_score column not found. Skipping feedback normalization.")
    feedback_exists = False

scaled_values = scaler.fit_transform(df[required_columns])

scaled_df = pd.DataFrame(
    scaled_values,
    columns=[col + "_norm" for col in required_columns]
)

df = pd.concat([df, scaled_df], axis=1)

# ----------------------------------
# 4. Performance Index
# ----------------------------------

if feedback_exists:
    df["performance_index_custom"] = (
        0.35 * df["task_completion_rate"] +
        0.20 * df["avg_attendance_norm"] +
        0.20 * df["avg_mentor_rating_norm"] +
        0.15 * df["avg_feedback_score_norm"] +
        0.10 * df["delay_score"]
    )
else:
    df["performance_index_custom"] = (
        0.40 * df["task_completion_rate"] +
        0.25 * df["avg_attendance_norm"] +
        0.25 * df["avg_mentor_rating_norm"] +
        0.10 * df["delay_score"]
    )

# ----------------------------------
# 5. Risk Flag
# ----------------------------------

def risk_flag(row):
    if (
        row["task_completion_rate"] < 0.60 or
        row["avg_attendance_norm"] < 0.50 or
        row["avg_mentor_rating_norm"] < 0.40
    ):
        return "High Risk"
    elif row["performance_index_custom"] > 0.80:
        return "High Performer"
    else:
        return "Average"

df["risk_flag"] = df.apply(risk_flag, axis=1)

# ----------------------------------
# Save Final Data
# ----------------------------------

df.to_csv("final_feature_engineered_data.csv", index=False)

print("✅ Feature Engineering Completed Successfully")

# Download output file
files.download("final_feature_engineered_data.csv")

Saving intern_level_cleaned.csv to intern_level_cleaned (1).csv
Uploaded file: intern_level_cleaned (1).csv
Available Columns:
 Index(['intern_id', 'name', 'gender', 'domain', 'college', 'avg_attendance',
       'avg_task_completion', 'total_tasks_assigned', 'total_tasks_completed',
       'avg_mentor_rating', 'total_late_submissions', 'weeks_completed',
       'performance_index'],
      dtype='object')
⚠ avg_feedback_score column not found. Skipping feedback normalization.
✅ Feature Engineering Completed Successfully


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>